# Long-Context TTFT Crossover Sweep

Companion to the T4 sweep in the paper's break-even section. That sweep used
TinyLlama-1.1B, whose 2,048-token context window capped it at 1,792 tokens, only
just past its predicted crossover of N* ~ 1,561. It could show the curve crossing
parity but not what happens well beyond it.

This notebook repeats the measurement on a long-context model so the sweep spans a
decade of context length entirely above N*, turning 'brackets the prediction' into
'tracks the prediction'.

**Runtime:** GPU (a free T4 suffices). **Expected time:** 15-25 minutes.

**Output:** `context_sweep_results.json` -- download it and hand it back.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install 'transformers>=4.44,<5' accelerate 2>&1 | tail -1


## Configuration

`Qwen2.5-0.5B` is the default because it is ungated (no HuggingFace token required)
and has a 32k context window. `Llama-3.2-1B` is a closer architectural match to the
paper's GQA discussion but is gated; select it only if this account has accepted its
license.


In [ ]:
MODEL = 'Qwen/Qwen2.5-0.5B'          # ungated, 32k context
# MODEL = 'meta-llama/Llama-3.2-1B'  # gated; requires HF login

CONTEXT_LENGTHS = [512, 2048, 4096, 8192, 16384]
TRIALS = 5      # per length; medians are reported


## Setup

Two measurement rules carried over from the paper, both easy to get wrong:

1. **Cold TTFT is a prefill-only forward pass**, never a `generate()` total.
   Comparing a cold prefill+decode against a warm first token inflates the apparent
   speedup by roughly 2.5x.
2. **Every timing is bracketed by `torch.cuda.synchronize()`**, or CUDA's
   asynchronous kernel launches make the GPU look arbitrarily fast.


In [ ]:
import torch, time, json, gc, os, tempfile, statistics as st
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), 'Switch the runtime to GPU'
device, dtype = 'cuda', torch.float16

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device).eval()

cfg = model.config
L = cfg.num_hidden_layers
Hq = cfg.num_attention_heads
Hkv = getattr(cfg, 'num_key_value_heads', Hq)
d = cfg.hidden_size // Hq
P = sum(p.numel() for p in model.parameters())
kv_per_token = 2 * L * Hkv * d * 2      # 2 tensors x FP16

print(f'{MODEL}: {L} layers, {Hq} Q-heads, {Hkv} KV-heads, head_dim={d}')
print(f'{P/1e9:.2f}B params, {kv_per_token/1024:.0f} KB KV per token')


## Predicted crossover

Generated here from the same expressions as the paper rather than copied, so the
prediction is reproducible from this notebook alone: prefill costs
`(2PN + 4*L*Hq*d*N^2) / (F*eta)` and restore moves `2*L*Hkv*d*b*N` bytes at the
tier's latency and bandwidth.


In [ ]:
T4_PEAK, T4_MFU = 65e12, 0.30      # dense FP16 FLOP/s; achievable prefill utilization
NVME_LAT, NVME_BW = 0.010, 2e9

def prefill_s(n):
    return (2 * P * n + 4 * L * Hq * d * n * n) / (T4_PEAK * T4_MFU)

def restore_s(n):
    return NVME_LAT + kv_per_token * n / NVME_BW

n_star = next(n for n in range(1, 200000) if prefill_s(n) > restore_s(n))
print(f'Predicted crossover N* = {n_star:,} tokens (T4, NVMe)')
lo, hi = min(CONTEXT_LENGTHS), max(CONTEXT_LENGTHS)
print(f'Sweep spans {lo}-{hi} tokens = {lo/n_star:.1f}x to {hi/n_star:.1f}x N*')


## Measurement

For each context length: prefill cold (timed), serialize the KV cache to disk,
reload it, resume from the restored cache (timed), and verify the resumed next-token
prediction matches the cold path.


In [ ]:
TMP = tempfile.mkdtemp()

def make_prompt(n_tokens):
    base = ('The tiered cache stores attention keys and values across a memory '
            'hierarchy so a resumed conversation need not be recomputed. ')
    unit = tok(base, return_tensors='pt').input_ids.shape[1]
    reps = max(1, n_tokens // unit + 2)
    ids = tok(base * reps, return_tensors='pt').input_ids[:, :n_tokens]
    return ids.to(device)

def save_kv(past, path):
    flat = {}
    if hasattr(past, 'key_cache'):
        pairs = list(zip(past.key_cache, past.value_cache))
    else:
        pairs = list(past)
    for i, (k, v) in enumerate(pairs):
        flat[str(i) + '_k'] = k.cpu()
        flat[str(i) + '_v'] = v.cpu()
    torch.save(flat, path)
    return os.path.getsize(path)

def load_kv(path, n_layers):
    flat = torch.load(path)
    keys = [flat[str(i) + '_k'].to(device) for i in range(n_layers)]
    vals = [flat[str(i) + '_v'].to(device) for i in range(n_layers)]
    legacy = tuple(zip(keys, vals))
    try:
        from transformers import DynamicCache
        return DynamicCache.from_legacy_cache(legacy)
    except Exception:
        return legacy


In [ ]:
results = []

for n in CONTEXT_LENGTHS:
    ids = make_prompt(n)
    actual = ids.shape[1]
    cold_ms, warm_ms, sizes, identical = [], [], [], []

    for _ in range(TRIALS):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        with torch.no_grad():
            out = model(ids, use_cache=True)
        torch.cuda.synchronize()
        cold_ms.append((time.perf_counter() - t0) * 1000)
        cold_tok = int(out.logits[:, -1, :].argmax(-1))

        path = os.path.join(TMP, 'kv.pt')
        sizes.append(save_kv(out.past_key_values, path))
        del out; gc.collect(); torch.cuda.empty_cache()

        torch.cuda.synchronize(); t1 = time.perf_counter()
        past = load_kv(path, L)
        with torch.no_grad():
            warm = model(ids[:, -1:], past_key_values=past, use_cache=True)
        torch.cuda.synchronize()
        warm_ms.append((time.perf_counter() - t1) * 1000)
        identical.append(int(warm.logits[:, -1, :].argmax(-1)) == cold_tok)
        del warm, past; gc.collect(); torch.cuda.empty_cache()

    row = {
        'tokens': actual,
        'cold_ttft_ms': st.median(cold_ms),
        'warm_path_ms': st.median(warm_ms),
        'speedup': st.median(cold_ms) / st.median(warm_ms),
        'kv_mb': st.median(sizes) / 1e6,
        'predicted_prefill_ms': prefill_s(actual) * 1000,
        'predicted_restore_ms': restore_s(actual) * 1000,
        'next_token_identical': str(sum(identical)) + '/' + str(len(identical)),
        'trials': TRIALS,
    }
    results.append(row)
    print('{:>6} tok | cold {:>8.1f}ms | warm {:>7.1f}ms | {:>5.2f}x | KV {:>6.1f}MB | id {}'.format(
        actual, row['cold_ttft_ms'], row['warm_path_ms'], row['speedup'],
        row['kv_mb'], row['next_token_identical']))


## Results


In [ ]:
summary = {
    'model': MODEL,
    'gpu': torch.cuda.get_device_name(0),
    'geometry': {'layers': L, 'q_heads': Hq, 'kv_heads': Hkv, 'head_dim': d,
                 'params': P, 'kv_bytes_per_token': kv_per_token},
    'predicted_n_star': n_star,
    'mfu_assumed': T4_MFU,
    'rows': results,
}
with open('context_sweep_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('{:>8}{:>12}{:>12}{:>10}'.format('tokens', 'measured', 'predicted', 'speedup'))
for r in results:
    print('{:>8}{:>10.0f}ms{:>10.0f}ms{:>9.2f}x'.format(
        r['tokens'], r['cold_ttft_ms'], r['predicted_prefill_ms'], r['speedup']))
print()
print('Saved context_sweep_results.json -- download this file.')


### What to look for

- **Speedup should rise monotonically** with context length and stay above 1x across
  the whole sweep, since every point here sits far above N*.
- **Measured vs predicted cold TTFT** should agree within roughly 2x. Systematic
  disagreement means the assumed MFU (0.30) needs recalibrating for this GPU, not
  that the model's shape is wrong.
- **`next_token_identical` should be 5/5** at long contexts. Occasional mismatches at
  short contexts are near-tie argmax flips caused by batched-vs-incremental kernel
  numerics, inherent to any cache reuse; the paper's in-memory control ablation shows
  the storage round-trip contributes nothing beyond that.
